# **Video Processing Week 2: Analisis Manusia (Wajah & Tubuh) dengan MediaPipe**

Minggu ini kita akan mempelajari **analisis manusia dalam video** menggunakan teknologi AI melalui MediaPipe. Fokus utama adalah deteksi dan tracking fitur wajah serta pose tubuh manusia secara real-time dengan akurasi tinggi.

**Tujuan Pembelajaran:**

Di akhir sesi ini kita akan mampu:
- Menggunakan model AI (via MediaPipe) untuk mengekstrak fitur wajah dan tubuh manusia secara real-time
- Mendeteksi wajah dan 468 titik landmark pada wajah dengan presisi tinggi
- Mengimplementasikan aplikasi deteksi kedipan mata menggunakan Eye Aspect Ratio (EAR)
- Mendeteksi 33 titik landmark pada tubuh manusia (skeleton/pose detection)
- Membangun aplikasi computer vision praktis untuk analisis gerakan manusia

**Topik Praktik:**
- **Pengenalan MediaPipe**: Setup dan penggunaan library MediaPipe untuk solusi AI real-time
- **Facial Landmark Detection**: Deteksi 468 titik wajah dan aplikasi deteksi kedipan mata
- **Pose Landmark Detection**: Deteksi skeleton tubuh dan analisis sudut sendi untuk deteksi gerakan

> *This module is inspired by the development of last semester’s materials.* 

## **Identitas Mahasiswa**

- **Nama:** Cidesta Mentari Marintan Simbolon  
- **NIM:** 122400096  
- **Kelas:** Pengolahan Citra

## **Pengenalan MediaPipe**

MediaPipe adalah framework open-source dari Google yang dirancang untuk membangun pipeline pemrosesan media secara real-time, seperti visi komputer dan pengolahan audio. MediaPipe menyediakan model deteksi wajah yang ringan, akurat, dan cepat, yang dapat digunakan baik untuk gambar statis maupun video streaming real-time.

*Sebelum lanjut, kita coba import library yang akan kita butuhkan dulu*

### **Menggunakan Mediapipe Real-time Face Mesh Detection**

In [12]:
import cv2
import os
import mediapipe as mp

PATH_VIDEO = os.path.join(os.getcwd(), 'data', 'man_walking.mp4')

mp_face = mp.solutions.face_detection

# FIX UTAMA: model lebih fleksibel + confidence diturunkan
face_det = mp_face.FaceDetection(
    model_selection=1,
    min_detection_confidence=0.2
)

cap = cv2.VideoCapture(PATH_VIDEO)

if not cap.isOpened():
    raise RuntimeError("Video tidak bisa dibuka")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    h, w = frame.shape[:2]

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = face_det.process(rgb)

    # debug kecil (biar kamu yakin detect)
    if result.detections:
        print("FACE DETECTED:", len(result.detections))

        for det in result.detections:
            box = det.location_data.relative_bounding_box

            x = int(box.xmin * w)
            y = int(box.ymin * h)
            bw = int(box.width * w)
            bh = int(box.height * h)

            x = max(0, x)
            y = max(0, y)

            x2 = min(x + bw, w)
            y2 = min(y + bh, h)

            # 🔥 PENANDA WAJAH (WAJIB KELIHATAN)
            cv2.rectangle(frame, (x, y), (x2, y2), (0, 255, 0), 3)

            cx = x + bw // 2
            cy = y + bh // 2

            cv2.circle(frame, (cx, cy), 6, (0, 0, 255), -1)

            cv2.putText(frame, "FACE", (x, y - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                        (255, 0, 0), 2)

    else:
        cv2.putText(frame, "NO FACE DETECTED", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1,
                    (0, 0, 255), 2)

    cv2.imshow("Face Detection FIXED", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED: 1
FACE DETECTED:

>>Program yang saya buat menggunakan OpenCV dan MediaPipe untuk mendeteksi wajah pada video secara real-time. Video dibaca frame per frame, lalu diubah dari BGR ke RGB agar bisa diproses oleh MediaPipe. Setelah itu sistem mendeteksi wajah menggunakan FaceDetection, kemudian jika wajah ditemukan akan muncul kotak (bounding box), titik tengah wajah, dan teks “FACE” pada bagian wajah. Jika tidak ada wajah yang terdeteksi, akan ditampilkan tulisan “NO FACE DETECTED”. Hasil akhirnya berupa video dengan penanda otomatis pada wajah yang terdeteksi.

## **Facial Landmark Detection**

MediaPipe Face Mesh dapat mendeteksi hingga 478 titik landmark pada wajah manusia dengan presisi tinggi, memungkinkan analisis detail fitur wajah seperti mata, hidung, mulut, dan kontur wajah untuk berbagai aplikasi seperti deteksi emosi, tracking mata, dan augmented reality.

Berikut adalah list landmark beserta posisinya: https://storage.googleapis.com/mediapipe-assets/documentation/mediapipe_face_landmark_fullsize.png

### **Deteksi wajah dan 468 titik landmark pada wajah**

In [14]:
import cv2
import mediapipe as mp

mp_mesh = mp.solutions.face_mesh
mp_draw = mp.solutions.drawing_utils
mp_style = mp.solutions.drawing_styles

mesh = mp_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

cam = cv2.VideoCapture(0)

if not cam.isOpened():
    raise RuntimeError("Webcam tidak bisa diakses")

while True:
    success, frame = cam.read()
    if not success:
        break

    h, w = frame.shape[:2]

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    output = mesh.process(rgb_frame)

    if output.multi_face_landmarks:
        for lm in output.multi_face_landmarks:

            mp_draw.draw_landmarks(
                image=frame,
                landmark_list=lm,
                connections=mp_mesh.FACEMESH_TESSELATION,
                connection_drawing_spec=mp_style.get_default_face_mesh_tesselation_style()
            )

            mp_draw.draw_landmarks(
                image=frame,
                landmark_list=lm,
                connections=mp_mesh.FACEMESH_CONTOURS,
                connection_drawing_spec=mp_style.get_default_face_mesh_contours_style()
            )

            mp_draw.draw_landmarks(
                image=frame,
                landmark_list=lm,
                connections=mp_mesh.FACEMESH_IRISES,
                connection_drawing_spec=mp_style.get_default_face_mesh_iris_connections_style()
            )

    cv2.putText(frame, "Face Mesh Active (press q)", (10, h - 15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.imshow("Face Mesh Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cam.release()
cv2.destroyAllWindows()
mesh.close()

>>Program ini menggunakan OpenCV dan MediaPipe Face Mesh untuk mendeteksi serta memetakan titik-titik wajah secara real-time melalui webcam. Pertama, webcam diakses menggunakan `cv2.VideoCapture(0)`, kemudian setiap frame yang diambil diubah dari format BGR ke RGB agar sesuai dengan input MediaPipe. Frame tersebut diproses menggunakan model Face Mesh untuk mendapatkan landmark wajah. Jika wajah terdeteksi, sistem akan menggambar struktur wajah secara detail menggunakan tiga jenis koneksi yaitu tesselation (kerangka wajah), contours (kontur wajah seperti mata dan bibir), serta iris (bagian mata) sehingga menghasilkan visualisasi wajah berbentuk titik-titik dan garis. Hasil akhir ditampilkan secara langsung pada jendela video sampai pengguna menekan tombol “q” untuk keluar.

### **Aplikasi sederhana: Deteksi Kedipan Mata (menggunakan rasio aspek mata / Eye Aspect Ratio)**

Aplikasi deteksi kedipan mata menggunakan Eye Aspect Ratio (EAR) bekerja dengan menghitung rasio antara jarak vertikal dan horizontal mata dari landmark wajah. Berikut tahapan implementasinya:

**Tahapan Implementasi:**

1. **Ekstraksi Koordinat Mata**: Mengambil 6 titik landmark khusus untuk setiap mata (kiri dan kanan) dari 468 landmark wajah MediaPipe
2. **Perhitungan EAR**: Menghitung Eye Aspect Ratio menggunakan rumus: `EAR = (|p2-p6| + |p3-p5|) / (2 * |p1-p4|)` dimana p1-p6 adalah 6 titik landmark mata
3. **Threshold Detection**: Membandingkan nilai EAR dengan threshold (biasanya ~0.25) - jika EAR di bawah threshold berarti mata tertutup
4. **Frame Counting**: Menghitung berapa frame berturut-turut mata tertutup untuk menghindari false positive
5. **Blink Counter**: Increment counter kedipan ketika mata kembali terbuka setelah tertutup dalam durasi yang wajar

**Kegunaan**: Aplikasi ini berguna untuk sistem monitoring kantuk pengemudi, kontrol perangkat hands-free, atau analisis perhatian dalam pembelajaran online.

**Nilai EAR Umum**
- **Mata Terbuka**: ~0.3 - 0.4
- **Mata Tertutup**: ~0.2 - 0.3

**Pemilihan Landmark**
Kode ini menggunakan indeks landmark MediaPipe Face Mesh yang spesifik:
- **Mata Kanan**: [33, 159, 158, 133, 153, 145]
- **Mata Kiri**: [362, 380, 374, 263, 386, 385]

In [17]:
import numpy as np
import cv2
import mediapipe as mp

# Fungsi untuk menghitung Eye Aspect Ratio (EAR)
def ear_calculator(eye_points):
    v1 = np.linalg.norm(eye_points[1] - eye_points[5])
    v2 = np.linalg.norm(eye_points[2] - eye_points[4])
    h = np.linalg.norm(eye_points[0] - eye_points[3])

    return (v1 + v2) / (2.0 * h)

# Landmark mata kiri dan kanan
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

# Parameter deteksi kedipan
THRESHOLD_EAR = 0.2
FRAME_LIMIT = 2

blink_count = 0
blink_total = 0

# Inisialisasi Face Mesh
mp_mesh = mp.solutions.face_mesh
mp_draw = mp.solutions.drawing_utils
mp_style = mp.solutions.drawing_styles

mesh = mp_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

>>Program ini digunakan untuk mendeteksi kedipan mata menggunakan metode Eye Aspect Ratio (EAR) dengan bantuan MediaPipe Face Mesh. Pertama, dibuat fungsi untuk menghitung nilai EAR berdasarkan jarak vertikal dan horizontal titik landmark pada mata, di mana nilai ini digunakan untuk menentukan apakah mata terbuka atau tertutup. Kemudian ditentukan indeks landmark untuk mata kiri dan kanan, serta nilai threshold untuk mendeteksi kedipan dan jumlah frame berturut-turut agar kedipan dapat dihitung lebih stabil. Setelah itu dilakukan inisialisasi MediaPipe Face Mesh untuk mendeteksi 468 titik wajah secara real-time. Hasil dari proses ini nantinya digunakan untuk menghitung jumlah kedipan berdasarkan perubahan nilai EAR pada setiap frame video atau webcam.

In [22]:
import cv2
import numpy as np
import mediapipe as mp

# ===== EAR FUNCTION =====
def eye_ratio(points):
    top_bottom_1 = np.linalg.norm(points[1] - points[5])
    top_bottom_2 = np.linalg.norm(points[2] - points[4])
    left_right = np.linalg.norm(points[0] - points[3])
    return (top_bottom_1 + top_bottom_2) / (2 * left_right)


# ===== LANDMARK INDEX =====
LEFT_EYE_IDS = [33, 160, 158, 133, 153, 144]
RIGHT_EYE_IDS = [362, 385, 387, 263, 373, 380]

# ===== PARAMETER =====
THRESH = 0.2
FRAME_LIMIT = 2

blink_frames = 0
blink_total = 0

# ===== MEDIAPIPE INIT =====
mp_face = mp.solutions.face_mesh

with mp_face.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_model:

    cam = cv2.VideoCapture(0)

    if not cam.isOpened():
        raise Exception("Camera tidak bisa dibuka")

    while cam.isOpened():

        ok, img = cam.read()
        if not ok:
            break

        h, w = img.shape[:2]
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        output = face_model.process(rgb)

        if output.multi_face_landmarks:

            for face in output.multi_face_landmarks:

                def extract_eye(indices):
                    pts = []
                    for i in indices:
                        lm = face.landmark[i]
                        pts.append(np.array([lm.x * w, lm.y * h]))
                        cv2.circle(img, (int(lm.x * w), int(lm.y * h)), 2, (0, 255, 0), -1)
                    return pts

                left_eye = extract_eye(LEFT_EYE_IDS)
                right_eye = extract_eye(RIGHT_EYE_IDS)

                ear_score = (eye_ratio(left_eye) + eye_ratio(right_eye)) / 2

                if ear_score < THRESH:
                    blink_frames += 1
                else:
                    if blink_frames >= FRAME_LIMIT:
                        blink_total += 1
                    blink_frames = 0

                state = "CLOSE" if ear_score < THRESH else "OPEN"
                color = (0, 0, 255) if state == "CLOSE" else (0, 255, 0)

                cv2.putText(img, f"EAR={ear_score:.2f}", (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

                cv2.putText(img, f"STATE={state}", (10, 60),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

                cv2.putText(img, f"BLINKS={blink_total}", (10, 90),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        cv2.imshow("Blink System (My Version)", img)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cam.release()
    cv2.destroyAllWindows()

print("Total blink detected:", blink_total)

Total blink detected: 4


>>Program ini digunakan untuk mendeteksi kedipan mata secara real-time menggunakan webcam dengan bantuan MediaPipe Face Mesh dan metode Eye Aspect Ratio (EAR). Video dari webcam diproses frame per frame, kemudian landmark wajah diekstrak khusus pada area mata kiri dan kanan berdasarkan indeks tertentu. Koordinat tersebut digunakan untuk menghitung nilai EAR yang menunjukkan kondisi mata terbuka atau tertutup. Jika nilai EAR berada di bawah threshold, sistem menganggap mata sedang berkedip dan menghitungnya berdasarkan jumlah frame berturut-turut. Hasil deteksi ditampilkan secara langsung berupa nilai EAR, status mata (OPEN/CLOSE), serta total kedipan pada layar video dengan visual titik pada area mata.

## **Pose Landmark Detection**

MediaPipe Pose Landmark Detection mendeteksi dan melacak 33 titik landmark pada tubuh manusia. memungkinkan komputer untuk "melihat" dan memahami postur serta gerakan tubuh manusia.

**33 Titik Landmark Tubuh:**
- **Wajah**: Hidung, mata kiri/kanan, telinga kiri/kanan (5 titik)
- **Tubuh Atas**: Bahu, siku, pergelangan tangan kiri/kanan (6 titik) 
- **Tubuh Tengah**: Pinggul kiri/kanan (2 titik)
- **Kaki**: Pinggul, lutut, pergelangan kaki, tumit, ujung kaki kiri/kanan (20 titik)

**Kegunaan Praktis:**
- **Analisis Olahraga**: Mengukur teknik gerakan atlet, mendeteksi postur yang salah
- **Fitness Apps**: Menghitung repetisi push-up, squat, atau latihan lainnya
- **Rehabilitasi Medis**: Monitoring progress pasien fisioterapi
- **Game & AR**: Kontrol karakter game menggunakan gerakan tubuh
- **Analisis Ergonomi**: Evaluasi postur kerja untuk mencegah cedera

### **Deteksi 33 titik landmark pada tubuh manusia (skeleton)**

In [26]:
import cv2
import mediapipe as mp

# model pose
pose_detector = mp.solutions.pose.Pose()
drawer = mp.solutions.drawing_utils

cam = cv2.VideoCapture(0)

if not cam.isOpened():
    raise Exception("Webcam gagal dibuka")

def process_frame(img):
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return pose_detector.process(rgb)

def draw_pose(img, result):
    if result.pose_landmarks:
        drawer.draw_landmarks(
            img,
            result.pose_landmarks,
            mp.solutions.pose.POSE_CONNECTIONS
        )

while True:
    ret, frame = cam.read()
    if not ret:
        break

    result = process_frame(frame)
    draw_pose(frame, result)

    cv2.putText(
        frame,
        "BODY TRACKING ON",
        (10, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 255),
        2
    )

    cv2.imshow("Pose Tracker (Custom Version)", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cam.release()
cv2.destroyAllWindows()

>>Program ini digunakan untuk mendeteksi pose tubuh manusia secara real-time menggunakan MediaPipe Pose dan OpenCV. Webcam digunakan untuk menangkap video, kemudian setiap frame diproses dengan mengubahnya ke format RGB sebelum dimasukkan ke model pose detection. Hasil deteksi berupa landmark tubuh akan digambar sebagai skeleton yang menghubungkan titik-titik tubuh seperti tangan, kaki, dan badan. Program dibuat menggunakan fungsi terpisah agar proses deteksi dan penggambaran lebih terstruktur. Hasil akhir ditampilkan secara langsung pada layar video hingga pengguna menekan tombol “q” untuk keluar.

### **Aplikasi sederhana: Menghitung Sudut Siku Kanan**

In [28]:
import numpy as np

def angle_between_points(p1, p2, p3):
    p1 = np.array(p1)
    p2 = np.array(p2)
    p3 = np.array(p3)

    vec1 = p1 - p2
    vec2 = p3 - p2

    dot_product = np.dot(vec1, vec2)
    mag_vec1 = np.linalg.norm(vec1)
    mag_vec2 = np.linalg.norm(vec2)

    cos_val = dot_product / (mag_vec1 * mag_vec2)

    cos_val = np.clip(cos_val, -1.0, 1.0)

    rad = np.arccos(cos_val)
    degree = np.degrees(rad)

    return degree

>>Fungsi ini digunakan untuk menghitung sudut antara tiga titik koordinat dalam bentuk derajat menggunakan konsep vektor. Tiga titik input diubah menjadi array NumPy, kemudian dibentuk dua vektor yaitu dari titik tengah ke titik pertama dan dari titik tengah ke titik ketiga. Nilai sudut dihitung menggunakan dot product kedua vektor dan dibagi dengan hasil perkalian panjang vektornya untuk mendapatkan nilai cosinus sudut. Nilai tersebut kemudian dibatasi agar tidak melebihi rentang -1 sampai 1 untuk menghindari error numerik. Setelah itu, nilai cosinus diubah menjadi radian menggunakan arccos, lalu dikonversi ke derajat sebagai hasil akhir sudut.

In [34]:
import cv2
import numpy as np
import mediapipe as mp

# ======================
# INIT MODEL
# ======================
pose = mp.solutions.pose.Pose(
    static_image_mode=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# ======================
# LANDMARK INDEX
# ======================
JOINTS = {
    "shoulder": 12,
    "elbow": 14,
    "wrist": 16
}

# ======================
# FUNCTION ANGLE
# ======================
def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)

    ba = a - b
    bc = c - b

    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    cosine = np.clip(cosine, -1.0, 1.0)

    return np.degrees(np.arccos(cosine))


# ======================
# EXTRACT POINTS
# ======================
def extract_points(lm, w, h):
    return {
        name: np.array(
            [lm[idx].x * w, lm[idx].y * h],
            dtype=np.int32   # 🔥 FIX UTAMA DI SINI
        )
        for name, idx in JOINTS.items()
    }


# ======================
# DRAW FUNCTION (FIX OPEN CV ERROR)
# ======================
def draw_arm(img, pts):
    s = tuple(pts["shoulder"])
    e = tuple(pts["elbow"])
    w = tuple(pts["wrist"])

    cv2.line(img, s, e, (255, 255, 255), 2)
    cv2.line(img, e, w, (255, 255, 255), 2)

    cv2.circle(img, s, 6, (0, 255, 0), -1)
    cv2.circle(img, e, 6, (0, 0, 255), -1)
    cv2.circle(img, w, 6, (0, 255, 0), -1)


# ======================
# CAMERA
# ======================
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Webcam tidak bisa dibuka")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    h, w = frame.shape[:2]

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = pose.process(rgb)

    if result.pose_landmarks:

        lm = result.pose_landmarks.landmark

        pts = extract_points(lm, w, h)

        angle = calculate_angle(
            pts["shoulder"],
            pts["elbow"],
            pts["wrist"]
        )

        draw_arm(frame, pts)

        cv2.putText(
            frame,
            f"Siku: {angle:.1f} deg",
            (10, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 255),
            2
        )

    cv2.imshow("Pose Angle FIXED", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
pose.close()

>>Program yang saya buat ini menggunakan OpenCV dan MediaPipe Pose untuk menghitung sudut siku kanan secara real-time lewat webcam. Pertama, video dari webcam diambil lalu setiap frame diproses dengan mengubahnya ke RGB supaya bisa dibaca oleh MediaPipe. Setelah itu sistem mendeteksi landmark tubuh, lalu saya ambil titik bahu, siku, dan pergelangan tangan kanan untuk dihitung koordinatnya dalam bentuk pixel. Dari tiga titik itu saya hitung sudut siku menggunakan rumus vektor dan trigonometri. Hasil akhirnya ditampilkan langsung di layar berupa nilai sudut siku beserta garis dan titik yang menunjukkan posisi lengan secara real-time sampai program dihentikan dengan tombol “q”.

### 🔎 Eksplorasi

- Cobalah mendeteksi bahu kiri
- Cobalah mendeteksi pose tubuh (berdiri, jongkok, tidur)

In [ ]:
### 🔎 Jawaban Eksplorasi (Bahu Kiri)

import cv2
import mediapipe as mp

mp_pose = mp.solutions.pose
pose = mp_pose.Pose()
mp_draw = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = pose.process(rgb)

    if result.pose_landmarks:
        lm = result.pose_landmarks.landmark

        # bahu kiri
        h, w, _ = frame.shape
        x = int(lm[11].x * w)
        y = int(lm[11].y * h)

        cv2.circle(frame, (x, y), 10, (0, 255, 0), -1)
        cv2.putText(frame, "Left Shoulder", (x+10, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

    cv2.imshow("Left Shoulder Detection", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [2]:
### 🔎 Jawaban Eksplorasi (Deteksi Pose)

import cv2
import mediapipe as mp

mp_pose = mp.solutions.pose
pose = mp_pose.Pose()
mp_draw = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    h, w, _ = frame.shape
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = pose.process(rgb)

    if result.pose_landmarks:
        lm = result.pose_landmarks.landmark

        shoulder_y = lm[11].y
        hip_y = lm[23].y
        knee_y = lm[25].y

        # klasifikasi sederhana
        if abs(shoulder_y - hip_y) < 0.1:
            pose_text = "Tidur / Horizontal"
        elif hip_y < knee_y:
            pose_text = "Berdiri"
        else:
            pose_text = "Jongkok"

        cv2.putText(frame, pose_text, (50, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1,
                    (0, 255, 0), 2)

    cv2.imshow("Pose Detection", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

>>Pose detection dilakukan menggunakan MediaPipe Pose dengan mendeteksi titik-titik tubuh seperti bahu, pinggul, dan lutut untuk menentukan posisi tubuh secara sederhana. Bahu kiri dapat dideteksi menggunakan landmark spesifik (11), sedangkan klasifikasi pose seperti berdiri, jongkok, dan tidur dilakukan berdasarkan perbandingan posisi vertikal antar titik tubuh. Sistem ini bekerja secara real-time melalui webcam dan cukup efektif untuk mengenali perubahan posisi tubuh secara umum.